In [ ]:
import getpass
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import data_pipeline
from src import visualization

- Leer el CSV

In [ ]:
df_combined = data_pipeline.load_csv(project_root / 'data' / 'df_combined.csv')

In [ ]:
df_combined

- Conectar a la base de datos

In [ ]:
password = os.getenv('MYSQL_PASSWORD') or getpass.getpass("Introduce la contraseña de MySQL: ")
engine = data_pipeline.create_db_connection(password=password)

- Insertar fechas y obtener id_date

In [ ]:
data_pipeline.insert_dates(df_combined, engine)
df_merged = data_pipeline.merge_dates(df_combined, engine)

- Dividir e insertar datos en las tablas 'assets' y 'economic_factors'

In [ ]:
data_pipeline.split_and_insert_data(df_merged, engine)

- Consulta 1: Crecimiento Anual de Bitcoin, Oro y S&P500 (2015-2024)

In [ ]:
consulta1 = """
SELECT
    year(d.date) AS year,
    a.price_bitcoin,
    a.price_gold,
    a.price_sp500,
    ROUND( (a.price_bitcoin - LAG(a.price_bitcoin) OVER (ORDER BY year(d.date))) / LAG(a.price_bitcoin) OVER (ORDER BY year(d.date)) * 100, 2) AS bitcoin_growth,
    ROUND( (a.price_gold - LAG(a.price_gold) OVER (ORDER BY year(d.date))) / LAG(a.price_gold) OVER (ORDER BY year(d.date)) * 100, 2) AS gold_growth,
    ROUND( (a.price_sp500 - LAG(a.price_sp500) OVER (ORDER BY year(d.date))) / LAG(a.price_sp500) OVER (ORDER BY year(d.date)) * 100, 2) AS sp500_growth
FROM 
    assets a
JOIN 
    dates d ON a.id_date = d.id_date
WHERE 
    d.date IN (SELECT MAX(d2.date) FROM dates d2 GROUP BY year(d2.date))
ORDER BY 
    year(d.date);
"""
df1 = data_pipeline.execute_query(engine, consulta1)
visualization.plot_growth_comparison(df1)

In [ ]:
df1

- Consulta 2: Promedio y dispersión mensual de precios de los activos

In [ ]:
consulta2 = """
SELECT 
    DATE_FORMAT(d.date, '%%Y-%%m') AS month_year,
    AVG(a.price_bitcoin) AS avg_bitcoin,
    STD(a.price_bitcoin) AS bitcoin_price_stddev,
    AVG(a.price_gold) AS avg_gold,
    STD(a.price_gold) AS gold_price_stddev,
    AVG(a.price_sp500) AS avg_sp500,
    STD(a.price_sp500) AS sp500_price_stddev
FROM 
    assets a
JOIN 
    dates d ON a.id_date = d.id_date
GROUP BY 
    month_year
ORDER BY 
    month_year;
"""

# Ejecutar la consulta y cargar los datos en un DataFrame
df2 = data_pipeline.execute_query(engine, consulta2)

# Mostrar el DataFrame resultante
df2

- Consulta 3: Efecto de las tasas de interés en el precio y la dispersión de precios de Bitcoin

In [ ]:
consulta3 = """
SELECT
    CASE
        WHEN e.interest_rate <= 2 THEN 'Bajas tasas de interés'
        ELSE 'Altas tasas de interés'
    END AS interest_rate_scenario,
    AVG(a.price_bitcoin) AS avg_bitcoin_price,
    STD(a.price_bitcoin) AS bitcoin_price_stddev
FROM
    economic_factors e
JOIN
    assets a ON e.id_date = a.id_date
WHERE
    e.interest_rate IS NOT NULL
GROUP BY
    interest_rate_scenario;
"""

# Ejecutar la consulta y cargar los datos en un DataFrame
df3 = data_pipeline.execute_query(engine, consulta3)

# Mostrar el DataFrame resultante
df3

- Consulta 4: Crecimiento del S&P 500 y su Relación con la Inflación (2015-2024)

In [ ]:
consulta4 = """
WITH year_end_prices AS (
    SELECT
        year(d.date) AS year,
        a.price_sp500
    FROM
        assets a
    JOIN
        dates d ON a.id_date = d.id_date
    WHERE
        d.date IN (SELECT MAX(d2.date) FROM dates d2 GROUP BY year(d2.date))
),
yearly_inflation AS (
    SELECT
        year(d.date) AS year,
        AVG(e.inflation) AS avg_inflation
    FROM
        economic_factors e
    JOIN
        dates d ON e.id_date = d.id_date
    GROUP BY
        year(d.date)
)
SELECT
    p.year,
    p.price_sp500,
    ROUND((p.price_sp500 - LAG(p.price_sp500) OVER (ORDER BY p.year)) / LAG(p.price_sp500) OVER (ORDER BY p.year) * 100, 2) AS sp500_growth,
    i.avg_inflation
FROM
    year_end_prices p
JOIN
    yearly_inflation i ON p.year = i.year
ORDER BY
    p.year;
"""
df4 = data_pipeline.execute_query(engine, consulta4)
visualization.plot_sp500_inflation(df4)

In [ ]:
df4

- Consulta 5: Promedio y desviación estándar de precios de Bitcoin en condiciones de alta volatilidad de mercado e inflación

In [ ]:
consulta5 = """
SELECT
    AVG(a.price_bitcoin) AS avg_bitcoin_price,
    STD(a.price_bitcoin) AS bitcoin_price_stddev,
    AVG(e.vix) AS avg_vix,
    AVG(e.inflation) AS avg_inflation
FROM
    assets a
JOIN
    economic_factors e ON a.id_date = e.id_date
WHERE
    e.vix > 30 AND e.inflation > 3;
"""

# Ejecutar la consulta y cargar los datos en un DataFrame
df5 = data_pipeline.execute_query(engine, consulta5)

# Mostrar el DataFrame resultante
df5